# Ola + Ollama dans Google Colab

Ce notebook installe **Ola** de bout en bout dans un runtime Google Colab : Ollama, le modèle par défaut `qwen3:1.7b`, Supervisor, cloudflared et un **Quick Tunnel Cloudflare** public.

**À faire avant de lancer :** dans le menu Colab, choisis si possible **Exécution → Modifier le type d'exécution → GPU** (par exemple T4). Le CPU fonctionne aussi, mais plus lentement.

> ⚠️ Le Quick Tunnel est **public et sans authentification** : toute personne qui connaît l'URL peut appeler l'API Ollama et consommer le runtime. Garde l'URL privée.

> Limite honnête de Colab : la machine virtuelle est jetable. Google Drive conserve les **modèles Ollama** et les **réglages `ola`** entre deux runtimes, mais les binaires, `/etc`, les processus et l'URL Quick Tunnel sont recréés à chaque nouveau runtime. Il suffit alors de relancer ce notebook depuis le début.

Lance les cellules **dans l'ordre**, de haut en bas.

## 1) Monter Google Drive et préparer l'état persistant

Colab demandera une autorisation Google Drive. C'est l'étape qui permet de retrouver les modèles et les réglages au prochain runtime. Sans Drive, le notebook continue en mode temporaire dans `/content/ola-colab`, mais rien n'est conservé après suppression du runtime.

In [ ]:
import os
from pathlib import Path

STATE_DIR = Path('/content/drive/MyDrive/ola-colab')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive monté :', STATE_DIR)
except Exception as exc:
    STATE_DIR = Path('/content/ola-colab')
    print('Google Drive non monté (', exc, ').')
    print('Mode temporaire : les modèles/réglages ne survivront pas au runtime.')

(STATE_DIR / 'config').mkdir(parents=True, exist_ok=True)
(STATE_DIR / 'models').mkdir(parents=True, exist_ok=True)

os.environ['OLA_COLAB'] = '1'
os.environ['OLA_COLAB_STATE_DIR'] = str(STATE_DIR)
os.environ['OLLAMA_MODELS'] = str(STATE_DIR / 'models')
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

print('OLA_COLAB =', os.environ['OLA_COLAB'])
print('État persistant =', os.environ['OLA_COLAB_STATE_DIR'])
print('Modèles Ollama =', os.environ['OLLAMA_MODELS'])

## 2) Vérifier le runtime et télécharger la version publiée de `ola`

Cette cellule affiche CPU/GPU/RAM/disque, télécharge l'artefact publié et vérifie son SHA-256 avant de le rendre exécutable.

In [ ]:
%%bash
set -e
cd /content
echo "Utilisateur : $(whoami)"
. /etc/os-release && echo "OS : $PRETTY_NAME"
echo "RAM disponible :"
free -h
echo "Disque du runtime :"
df -h / /content | tail -n +2
if command -v nvidia-smi >/dev/null 2>&1; then
  echo "GPU détecté :"
  nvidia-smi || true
else
  echo "Aucun GPU NVIDIA détecté : Ollama utilisera le CPU."
fi
BASE='https://raw.githubusercontent.com/JCVERSA/Ssh.jc/arena/e89a71aa-ssh-jc'
curl -fsSLO "$BASE/ollama-cloudflare-setup.sh"
curl -fsSLO "$BASE/ollama-cloudflare-setup.sh.sha256"
sha256sum -c ollama-cloudflare-setup.sh.sha256
chmod +x ollama-cloudflare-setup.sh
echo 'Artefact ola téléchargé et vérifié.'

## 3) Installation complète : Ollama + modèle + Quick Tunnel

Cette cellule utilise le setup rapide du projet : modèle intégré par défaut `qwen3:1.7b` + Quick Tunnel. Elle peut prendre plusieurs minutes, surtout au premier runtime (téléchargement d'Ollama, du modèle et de la TUI optionnelle). Si les modèles sont déjà dans Google Drive, Ollama les réutilise.

In [ ]:
%%bash
set -e
cd /content
./ollama-cloudflare-setup.sh setup default
echo
echo '--- État Colab ---'
ola colab status
echo
echo '--- Doctor ---'
ola doctor
echo
echo '--- Test de bout en bout ---'
ola test
echo
echo "URL PUBLIQUE : $(cat /etc/ollama-cloudflare-url)"

## 4) Tester l'URL publique depuis le notebook

La cellule suivante lit l'URL Quick Tunnel enregistrée par `ola`, appelle `/api/version`, liste les modèles et envoie une mini-conversation à `/api/chat`. Sur CPU, la première réponse peut prendre un peu de temps (chargement du modèle).

In [ ]:
import json
import urllib.request
from pathlib import Path

base_url = Path('/etc/ollama-cloudflare-url').read_text().strip()
print('URL publique :', base_url)

def call_json(url, payload=None, timeout=300):
    data = None
    headers = {}
    if payload is not None:
        data = json.dumps(payload).encode('utf-8')
        headers['Content-Type'] = 'application/json'
    req = urllib.request.Request(url, data=data, headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode('utf-8'))

version = call_json(base_url + '/api/version', timeout=60)
print('Version Ollama :', version)

tags = call_json(base_url + '/api/tags', timeout=60)
print('Modèles installés :', [m.get('name') for m in tags.get('models', [])])

chat = call_json(base_url + '/api/chat', {
    'model': 'qwen3:1.7b',
    'messages': [{'role': 'user', 'content': 'Réponds seulement : OK'}],
    'stream': False,
    'think': False,
})
print('Réponse du modèle :', chat.get('message', {}).get('content', '').strip())

## 5) Sauvegarder l'état et diagnostiquer plus tard

Le setup sauvegarde déjà le modèle par défaut, le mode Pack Speed et le keep-alive dans Google Drive. Relance cette cellule après avoir changé un réglage important (par exemple `ola speed`, `ola fluid` ou le modèle par défaut) pour resynchroniser ces petits fichiers. Les modèles téléchargés par Ollama sont déjà écrits directement dans Drive via `OLLAMA_MODELS`.

In [ ]:
%%bash
set -e
ola colab save
echo
ola status
echo
echo "URL PUBLIQUE ACTUELLE : $(cat /etc/ollama-cloudflare-url)"
echo "État persistant : ${OLA_COLAB_STATE_DIR:-non défini}"

## Commandes utiles pendant la session

Dans une nouvelle cellule `%%bash`, tu peux utiliser :

```bash
ola status       # état rapide
ola doctor       # diagnostic complet
ola test         # test API + modèle
ola logs         # journaux récents
ola recover      # relancer proprement les services du runtime actuel
ola colab status # CPU/GPU, Drive et limites du runtime
ola stop         # arrêter seulement le tunnel public
```

La TUI `ola tui` demande un vrai terminal interactif ; dans les cellules Colab, préfère les commandes texte ci-dessus.

## Au prochain runtime Colab

1. rouvre ce notebook ;
2. choisis à nouveau CPU ou GPU ;
3. relance toutes les cellules depuis le début ;
4. récupère la **nouvelle** URL Quick Tunnel affichée par la cellule d'installation.

Les modèles présents dans Google Drive n'ont pas à être retéléchargés, et les réglages `ola` sauvegardés sont restaurés par le setup. Les binaires et les services sont reconstruits dans le nouveau runtime.